#### Import Libraries

In [1]:
# Environment
from dotenv import load_dotenv
import os

# PDF Loader
from langchain_community.document_loaders import PyPDFLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# LCEL Components
from langchain_core.runnables import RunnablePassthrough

# Output Parser
from langchain_core.output_parsers import StrOutputParser

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

C:\Users\DELL\AppData\Local\Temp\ipykernel_3212\3361703605.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


#### Load API Key

In [2]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
print("API Loaded Successfully")

API Loaded Successfully


#### Initialize Gemini

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.3)
print("Gemini Ready")

Gemini Ready


#### Load PDF

In [4]:
pdf_path = (r"F:\GenAI\datasets\1.Introduction to sequential data and RNNs.pdf")
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("Pages:", len(documents))

Pages: 2


#### Chunking

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))

Chunks: 8


#### Create Embedding Model

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\DELL\AppData\Local\Temp\ipykernel_3212\412152783.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Create FAISS Vector Store

In [7]:
vector_store = FAISS.from_documents(chunks,embedding_model)
print("FAISS Ready")

FAISS Ready


####  Create Retriever

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

#### Create Prompt Template

In [9]:
prompt = ChatPromptTemplate.from_template(
"""
You are an AI Tutor.

Answer ONLY using the provided context.

Context:
{context}

Question:
{question}
"""
)

#### Context Formatter

In [10]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
)

#### Build LCEL Chain

In [11]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

#### Ask Question

In [12]:
response = rag_chain.invoke("What is RNN?")
print(response)

A Recurrent Neural Network (RNN) is a special type of neural network designed to process sequential data, where the order of information is important. Unlike traditional feedforward networks, RNNs have connections that loop back, allowing information from previous steps to influence the current output. This looping structure helps the network retain a form of memory, enabling it to learn.
